# 07 - Composite Loss Landscape

This notebook assembles a multi-term composite `Loss` and maps its value over
a two-dimensional sweep of the prediction's centre and width while the target
is held fixed. The goal is to confirm the composite objective has its global
minimum exactly at the true parameters and is smooth and well-behaved around
it, which is what makes it trainable by gradient descent.

Modules exercised: full `pipelines.training_pipeline.loss.Loss` with several
active terms and their per-term normalisation factors.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED   = 0
DEVICE = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "savefig.dpi"    : 300,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

print("repo root :", REPO_ROOT)
print("torch     :", torch.__version__)
print("device    :", DEVICE)


In [ ]:
from configuration.training_config import GaussianConfig, GeometryConfig, LossConfig
from pipelines.training_pipeline.loss import Loss
from tools import NullLogger, NullTracker

class IdentityNormStats:
    def normalize_output(self, tensor):
        return tensor

    def denormalize_output(self, tensor):
        return tensor

X_MIN, X_MAX = -10.0, 40.0
x_axis       = torch.linspace(X_MIN, X_MAX, 64, dtype=torch.float32)
gaussian_cfg = GaussianConfig(n_default_gaussians=1, x_min=X_MIN, x_max=X_MAX)

loss_cfg = LossConfig(
    use_mse_curve   = True, weight_mse_curve   = 1.0,
    use_param_l1    = True, weight_param_l1    = 0.5,
    use_cosine_curve= True, weight_cosine_curve= 0.5,
)
criterion = Loss(x_axis, NullLogger(), NullTracker(), gaussian_cfg,
                 loss_cfg, IdentityNormStats(), GeometryConfig())

TRUE_AMP, TRUE_MU, TRUE_SIG = 2.0, 12.0, 5.0

def params(amp, mu, sig, H=2, W=2):
    p = torch.tensor([amp, mu, sig], dtype=torch.float32).reshape(1, 3, 1, 1)
    return p.expand(1, 3, H, W).contiguous()

target = params(TRUE_AMP, TRUE_MU, TRUE_SIG)


## Two-dimensional loss surface over (centre, width)

We sweep the predicted centre and width on a grid, keeping amplitude at its
true value, and evaluate the total composite loss at each grid point.

In [ ]:
mu_grid  = torch.linspace(2.0, 22.0, 60)
sig_grid = torch.linspace(1.5, 10.0, 60)

Z = np.zeros((sig_grid.numel(), mu_grid.numel()), dtype=np.float32)
for i, sg in enumerate(sig_grid):
    for j, mu in enumerate(mu_grid):
        out     = criterion(params(TRUE_AMP, float(mu), float(sg)), target)
        Z[i, j] = float(out['total_loss'])

min_idx = np.unravel_index(np.argmin(Z), Z.shape)
best_mu, best_sig = float(mu_grid[min_idx[1]]), float(sig_grid[min_idx[0]])

fig, ax = plt.subplots(figsize=(6.5, 5))
cf = ax.contourf(mu_grid.numpy(), sig_grid.numpy(), Z, levels=30, cmap='viridis')
ax.contour(mu_grid.numpy(), sig_grid.numpy(), Z, levels=12, colors='white', linewidths=0.4, alpha=0.6)
ax.plot(TRUE_MU, TRUE_SIG, marker='*', color='red',    ms=16, label='true parameters')
ax.plot(best_mu, best_sig, marker='o', color='white',  ms=7, mfc='none', label='grid minimum')
ax.set_xlabel('predicted centre mu [m]')
ax.set_ylabel('predicted width sigma [m]')
ax.set_title('Composite loss landscape')
ax.legend(frameon=False, loc='upper right')
fig.colorbar(cf, ax=ax, label='total loss')
fig.tight_layout()
plt.show()

print('true   (mu, sigma) = (%.2f, %.2f)' % (TRUE_MU, TRUE_SIG))
print('argmin (mu, sigma) = (%.2f, %.2f)' % (best_mu, best_sig))


## One-dimensional slices through the true minimum

Slicing the surface along each axis through the true parameters gives a
clearer view of the basin shape and the location of the minimum.

In [ ]:
mu_slice  = [float(criterion(params(TRUE_AMP, float(mu), TRUE_SIG), target)['total_loss']) for mu in mu_grid]
sig_slice = [float(criterion(params(TRUE_AMP, TRUE_MU, float(sg)), target)['total_loss']) for sg in sig_grid]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(mu_grid.numpy(), mu_slice); axes[0].axvline(TRUE_MU, color='red', ls='--', lw=1)
axes[0].set_xlabel('centre mu [m]'); axes[0].set_title('slice at true sigma')
axes[1].plot(sig_grid.numpy(), sig_slice); axes[1].axvline(TRUE_SIG, color='red', ls='--', lw=1)
axes[1].set_xlabel('width sigma [m]'); axes[1].set_title('slice at true mu')
for ax in axes:
    ax.set_ylabel('total loss')
fig.tight_layout()
plt.show()


## Expected visual outcome

The contour map should show a single convex-looking basin whose minimum (red
star) coincides with the grid argmin (white circle) at the true parameters.
The one-dimensional slices are U-shaped with their minima on the red dashed
lines. This confirms the composite loss is correctly centred on the target
and provides a useful descent direction everywhere in the swept region.